Landing → Gate 1 (cổng kiểm tra lô)

Gate 1 là **Python thuần**, không cần Spark: chỉ đọc manifest, header và đếm dòng. Cả 5 luật đều
**REJECT cả lô** — lô bị từ chối nằm nguyên ở landing, pipeline không tự sửa.

In [1]:
import csv, json, logging, shutil, sys
from pathlib import Path

import pandas as pd

# Thư mục gốc dự án = nơi có src/config.py (kernel có thể mở với cwd bất kỳ)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
for p in (ROOT, ROOT / "tests"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src import config, validate

logging.basicConfig(level=logging.INFO, format="%(levelname)-5s %(name)-7s %(message)s",
                    stream=sys.stdout, force=True)
logging.getLogger("py4j").setLevel(logging.WARNING)
pd.set_option("display.max_columns", 40)

SANDBOX = ROOT / "notebook" / "_sandbox"             # mọi thứ notebook ghi ra nằm ở đây
SB_CASES = SANDBOX / "cases"                         # bản sao lô bị làm hỏng có chủ đích

Hàm tiện ích dùng trong notebook này:

In [2]:
def fresh(*paths: Path) -> None:
    """Xoá các thư mục/file con của sandbox — notebook chỉ dọn phần của mình."""
    for p in paths:
        if SANDBOX not in p.parents:
            raise ValueError(f"{p} nằm ngoài sandbox — không xoá")
        if p.is_dir():
            shutil.rmtree(p)
        elif p.exists():
            p.unlink()
    SANDBOX.mkdir(parents=True, exist_ok=True)

fresh(SB_CASES)
fresh                                  # notebook này chỉ ghi vào _sandbox/cases/

<function __main__.fresh(*paths: pathlib.Path) -> None>

## Landing

In [3]:
batches = validate.list_batches(config.LANDING_DIR)
print(f"{len(batches)} lô: {batches[0].name} → {batches[-1].name}")

20 lô: as_of_month=2025-01 → as_of_month=2026-08


In [4]:
# kiểm tra thông tin lô 2026 - 06
b_2606 = config.LANDING_DIR / "as_of_month=2026-06" 
manifest = validate.load_manifest(b_2606)
print("File trong", b_2606.name, ":", sorted(p.name for p in b_2606.iterdir()))
{k: manifest[k] for k in validate.MANIFEST_FIELDS}   

File trong as_of_month=2026-06 : ['_manifest.json', 'business_vehicle_valuation_202606.csv']


{'batch_id': 'bvv_202606_01',
 'as_of_month': '2026-06',
 'record_count': 3394,
 'file_name': 'business_vehicle_valuation_202606.csv',
 'file_sha256': '0021a3651a17bd17b3bfd3fc3f28f64515e7dd238711dd6bfd8ed527768bf510',
 'encoding': 'utf-8-sig',
 'delimiter': ','}

kiểm 5 luật validator trên một lô

In [5]:
data_file = b_2606 / manifest["file_name"]
data_file

WindowsPath('D:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/data/landing/business_vehicle_valuation/as_of_month=2026-06/business_vehicle_valuation_202606.csv')

In [6]:
actual_sha = validate.sha256_of(data_file)
actual_sha

'0021a3651a17bd17b3bfd3fc3f28f64515e7dd238711dd6bfd8ed527768bf510'

In [7]:
with open(data_file, newline="", encoding=manifest["encoding"]) as f:
    reader = csv.DictReader(f, delimiter=manifest["delimiter"])
    header, rows = reader.fieldnames, list(reader)
folder = validate.partition_value(b_2606)
col_values = {r["as_of_month"] for r in rows}
print("validate_batch():", validate.validate_batch(b_2606))
# [] lô không lỗi

validate_batch(): []


In [8]:
pd.DataFrame([
    ("MANIFEST_INVALID", all(k in manifest for k in validate.MANIFEST_FIELDS) and data_file.exists(),
     "đủ 7 cột bắt buộc, thấy file dữ liệu"),
    ("CHECKSUM_MISMATCH", actual_sha == manifest["file_sha256"],
     f"file {actual_sha[:16]}…  vs  manifest {manifest['file_sha256'][:16]}…"), #kiểm tra file có bị đổi không
    ("SCHEMA_COLUMN_MISSING", set(config.EXPECTED_COLUMNS) <= set(header),
     f"header {len(header)} cột, contract cần {len(config.EXPECTED_COLUMNS)}"), #kiểm tra schema có bị đổi không
    ("COUNT_MISMATCH", len(rows) == int(manifest["record_count"]),
     f"file {len(rows)} dòng, manifest khai {manifest['record_count']}"),       #kiểm tra record
    ("PARTITION_MISMATCH", manifest["as_of_month"] == folder and col_values == {folder},
     f"thư mục={folder}, manifest={manifest['as_of_month']}, cột={sorted(col_values)}"), #kiểm tra thư mục có đổi không
], columns=["rule_id", "đạt", "so sánh"])

,rule_id,đạt,so sánh
0,MANIFEST_INVALID,True,"đủ 7 cột bắt buộc, thấy file dữ liệu"
1,CHECKSUM_MISMATCH,True,file 0021a3651a17bd17… vs manifest 0021a3651...
2,SCHEMA_COLUMN_MISSING,True,"header 16 cột, contract cần 16"
3,COUNT_MISMATCH,True,"file 3394 dòng, manifest khai 3394"
4,PARTITION_MISMATCH,True,"thư mục=2026-06, manifest=2026-06, cột=['2026-..."


Kiểm tra 18 lô đã nạp

In [9]:
results = validate.validate_all(config.LANDING_DIR)
gate = pd.DataFrame([{"lô": p.name, "kết quả": "REJECT" if f else "OK",
                      "rule_id": ", ".join(x.rule_id for x in f)} for p, f in results.items()])
print(gate["kết quả"].value_counts().to_dict())
gate

{'OK': 20}


,lô,kết quả,rule_id
0,as_of_month=2025-01,OK,
1,as_of_month=2025-02,OK,
2,as_of_month=2025-03,OK,
3,as_of_month=2025-04,OK,
4,as_of_month=2025-05,OK,
5,as_of_month=2025-06,OK,
6,as_of_month=2025-07,OK,
7,as_of_month=2025-08,OK,
8,as_of_month=2025-09,OK,
9,as_of_month=2025-10,OK,


Sao chép lô `2026-06` vào `_sandbox/cases/`, mỗi bản làm hỏng **một kiểu** sao cho chỉ **một** luật bị
vi phạm (sau khi sửa file, `reseal` tính lại sha256 để lỗi khác không bị `CHECKSUM_MISMATCH` che mất).

In [10]:
def make_case(case: str, folder: str | None = None) -> Path:
    dst = SB_CASES / case / (folder or b_2606.name)
    shutil.copytree(b_2606, dst)
    return dst

test_cases = {}

In [11]:
# MANIFEST_INVALID: lô đưa vào không có manifest
d = make_case("MANIFEST_INVALID")
(d / "_manifest.json").unlink()
test_cases["MANIFEST_INVALID"] = d
test_cases

{'MANIFEST_INVALID': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/MANIFEST_INVALID/as_of_month=2026-06')}

In [12]:
def data_path(batch_dir: Path) -> Path:
    return batch_dir / validate.load_manifest(batch_dir)["file_name"]

def tamper_one_cell(batch_dir: Path) -> None:
    """Sửa tay một ô sau khi bàn giao (số dòng, header không đổi) -> chỉ checksum lệch."""
    f = data_path(batch_dir)
    f.write_bytes(f.read_bytes().replace(b",VND,", ",VNĐ,".encode("utf-8"), 1))

# CHECKSUM_MISMATCH: sửa tay một ô sau khi bàn giao
d = make_case("CHECKSUM_MISMATCH")
tamper_one_cell(d)
test_cases["CHECKSUM_MISMATCH"] = d
test_cases

{'MANIFEST_INVALID': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/MANIFEST_INVALID/as_of_month=2026-06'),
 'CHECKSUM_MISMATCH': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/CHECKSUM_MISMATCH/as_of_month=2026-06')}

In [13]:
# SCHEMA_COLUMN_MISSING — truy vấn trích bị đổi, rơi mất cột loai_dinh_gia
d = make_case("SCHEMA_COLUMN_MISSING")
f = data_path(d)
with open(f, newline="", encoding="utf-8-sig") as fh:
    src_rows = list(csv.DictReader(fh))
with open(f, "w", newline="", encoding="utf-8-sig") as fh:
    w = csv.DictWriter(fh, fieldnames=[c for c in src_rows[0] if c != "loai_dinh_gia"],
                       extrasaction="ignore")
    w.writeheader()
    w.writerows(src_rows)

def edit_manifest(batch_dir: Path, **changes) -> None:
    m = validate.load_manifest(batch_dir)
    m.update(changes)
    (batch_dir / "_manifest.json").write_text(json.dumps(m, ensure_ascii=False, indent=2),
                                              encoding="utf-8")

def reseal(batch_dir: Path) -> None:
    """Ghi lại sha256 đúng vào manifest — để lỗi khác không bị CHECKSUM_MISMATCH che mất."""
    edit_manifest(batch_dir, file_sha256=validate.sha256_of(data_path(batch_dir)))
    
reseal(d)
test_cases["SCHEMA_COLUMN_MISSING"] = d
test_cases

{'MANIFEST_INVALID': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/MANIFEST_INVALID/as_of_month=2026-06'),
 'CHECKSUM_MISMATCH': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/CHECKSUM_MISMATCH/as_of_month=2026-06'),
 'SCHEMA_COLUMN_MISSING': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/SCHEMA_COLUMN_MISSING/as_of_month=2026-06')}

In [14]:
# COUNT_MISMATCH — truyền file bị cụt mất dòng cuối
d = make_case("COUNT_MISMATCH")
f = data_path(d)
raw = f.read_bytes().rstrip(b"\r\n")
f.write_bytes(raw[: raw.rfind(b"\n") + 1])
reseal(d)
test_cases["COUNT_MISMATCH"] = d
test_cases

{'MANIFEST_INVALID': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/MANIFEST_INVALID/as_of_month=2026-06'),
 'CHECKSUM_MISMATCH': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/CHECKSUM_MISMATCH/as_of_month=2026-06'),
 'SCHEMA_COLUMN_MISSING': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/SCHEMA_COLUMN_MISSING/as_of_month=2026-06'),
 'COUNT_MISMATCH': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/COUNT_MISMATCH/as_of_month=2026-06')}

In [15]:
# PARTITION_MISMATCH — đặt nhầm lô tháng 6 vào thư mục tháng 5
test_cases["PARTITION_MISMATCH"] = make_case("PARTITION_MISMATCH", folder="as_of_month=2026-05")

{k: v.relative_to(SANDBOX).as_posix() for k, v in test_cases.items()}
test_cases

{'MANIFEST_INVALID': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/MANIFEST_INVALID/as_of_month=2026-06'),
 'CHECKSUM_MISMATCH': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/CHECKSUM_MISMATCH/as_of_month=2026-06'),
 'SCHEMA_COLUMN_MISSING': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/SCHEMA_COLUMN_MISSING/as_of_month=2026-06'),
 'COUNT_MISMATCH': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/COUNT_MISMATCH/as_of_month=2026-06'),
 'PARTITION_MISMATCH': WindowsPath('d:/NhanTran/NhanFiris/AIO_TOPIC/Topic/gia_tri_xe_oto/notebook/_sandbox/cases/PARTITION_MISMATCH/as_of_month=2026-05')}

In [16]:
check = []
for expected_rule, d in test_cases.items():
    findings = validate.validate_batch(d)
    check.append({"tình huống": expected_rule, "rule_id nhận được": [x.rule_id for x in findings],
                  "thông điệp": "; ".join(x.message for x in findings)})
check = pd.DataFrame(check)
check["khớp"] = [got == [exp] for exp, got in zip(check["tình huống"], check["rule_id nhận được"])]

assert check["khớp"].all(), "Có luật bắt sai lỗi"
assert validate.validate_batch(b_2606) == [], "Lô gốc ở landing phải còn nguyên"
check

,tình huống,rule_id nhận được,thông điệp,khớp
0,MANIFEST_INVALID,[MANIFEST_INVALID],không đọc được _manifest.json: [Errno 2] No su...,True
1,CHECKSUM_MISMATCH,[CHECKSUM_MISMATCH],sha256 796d8c4bba51… ≠ manifest 0021a3651a17…,True
2,SCHEMA_COLUMN_MISSING,[SCHEMA_COLUMN_MISSING],thiếu cột ['loai_dinh_gia'],True
3,COUNT_MISMATCH,[COUNT_MISMATCH],"tệp có 3393 record, manifest khai báo 3394",True
4,PARTITION_MISMATCH,[PARTITION_MISMATCH],"thư mục=2026-05, manifest=2026-06, cột as_of_m...",True
